1. 라이브러리 임포트

In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score, precision_recall_fscore_support
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input

2. 데이터 불러오기

In [17]:
df = pd.read_csv("scaled.csv")
print(f"원본 데이터 shape: {df.shape}")

원본 데이터 shape: (284807, 31)


3. 'Class' 결측치 확인 및 제거

In [18]:
# 3-1. (확인) 'Class' 열에 NaN이 있는지 확인
nan_count = df['Class'].isnull().sum()
if nan_count > 0:
    print(f"경고: 'Class' 열에 {nan_count}개의 NaN(결측치)이 있습니다. 이 행들을 제거합니다.")
    # 3-2. (해결) 'Class' 열에 NaN이 있는 행(row)을 제거
    df.dropna(subset=['Class'], inplace=True)

4. 입력(X), 출력(y) 분리

In [19]:
X = df.drop(columns=['Class'])
y = df['Class']

# 이미 scaled.csv는 스케일링 완료된 상태이므로 StandardScaler 불필요

5. Train/Test Split

In [20]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train/Test 분리 완료")

Train/Test 분리 완료


6. 클래스 가중치 계산

In [21]:
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weights = {0: class_weights[0], 1: class_weights[1]}
print(f"클래스 가중치: {class_weights}")

클래스 가중치: {0: np.float64(0.5008661206149896), 1: np.float64(289.14340101522845)}


7. MLP 모델 정의

In [22]:
def build_mlp(input_dim):
    model = Sequential([
        Input(shape=(input_dim,)),
        Dense(128, activation='relu'),
        BatchNormalization(),
        Dropout(0.3),

        Dense(64, activation='relu'),
        BatchNormalization(),
        Dropout(0.3),

        Dense(32, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy'
    )
    return model

8. 모델 학습

In [23]:
model = build_mlp(X_train.shape[1])
history = model.fit(
    X_train, y_train,
    epochs=8,
    batch_size=512,
    validation_split=0.2,
    verbose=1,
    class_weight=class_weights
)

Epoch 1/8
357/357 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - loss: 0.2583 - val_loss: 0.1578
Epoch 2/8
357/357 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.1723 - val_loss: 0.1037
Epoch 3/8
357/357 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - loss: 0.1358 - val_loss: 0.0886
Epoch 4/8
357/357 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.1492 - val_loss: 0.0864
Epoch 5/8
357/357 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.1083 - val_loss: 0.0708
Epoch 6/8
357/357 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.1074 - val_loss: 0.0625
Epoch 7/8
357/357 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.1069 - val_loss: 0.0723
Epoch 8/8
357/357 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.0985 - val_loss: 0.0681


9. 예측 및 평가

In [26]:
from sklearn.metrics import precision_score, recall_score  # ← 이 줄 추가!

y_pred_prob = model.predict(X_test)
threshold = 0.5
y_pred = (y_pred_prob > threshold).astype(int)

recall = recall_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)

print("\n모델 평가 결과 (MLP, threshold=0.5)")
print(f"Threshold = {threshold}")
print(f"Recall: {recall:.4f}")
print(f"Precision: {precision:.4f}")

1781/1781 ━━━━━━━━━━━━━━━━━━━━ 2s 920us/step

모델 평가 결과 (MLP, threshold=0.5)
Threshold = 0.5
Recall: 0.9082
Precision: 0.0646
